# Judge calibration mechanics

*Companion notebook for the Ciphero ML Engineer interview prep — anchors the
"how do you know a judge's `0.7` is really a 0.7?" question.*

When an LLM acts as a verifier judge (policy compliance, factuality, code review),
the model emits both a verdict and a confidence score. **Calibration** asks whether
those confidences match observed frequencies: of the cases the judge says are
`0.7`-likely positive, do ~70% turn out positive?

This notebook compares two Claude judge configurations on a labeled binary task:

1. **Well-prompted** — system prompt explicitly asks for frequency-faithful
   confidence ("if you say 0.7 it should be right ~70% of the time").
2. **Over-confident** — system prompt rewards decisive yes/no ("be confident,
   commit to clear decisions"), which we predict will skew probabilities high.

We score both with **Brier**, **ECE-10 (equal-mass)**, and a **reliability diagram**.
The bottom of the notebook poses the open question for Ciphero: what holdout do you
calibrate against — deterministic-labeled gold, or a policy-spec-as-target?

## Setup

**Dataset.** A 150-example balanced subset of IMDB sentiment
(`stanfordnlp/imdb`, label 0=negative, 1=positive). IMDB is a calibration-mechanics
proxy — the *methodology* (Brier / ECE / reliability) ports unchanged to a policy-
compliance verifier; the public dataset just lets the notebook ship reproducibly
without leaking customer policy material.

**Judge.** `claude-sonnet-4-6` via the official Anthropic Python SDK
(`client.messages.create`). Each call returns a JSON-shaped reply
`{"label": "positive"|"negative", "confidence": float in [0, 1]}`. We parse, then
flip `confidence` into the *probability the example is positive* (so it lines up
with the gold label space).

**Caching.** Every judge response is written to
`notebooks/cache/judge_responses_{config}.json` keyed by example index. Re-running
the notebook reads cache and skips API calls. Cell 6b clears the cache if you want
a fresh run.

**Cost guardrail.** 150 examples × 2 configs = 300 API calls, max ~150 input tokens
each. Budget is small; the cache makes a re-run free.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal, TypedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from anthropic import Anthropic
from datasets import load_dataset

# Local module — implements Brier / ECE / reliability binning from scratch.
from llm_eval.calibration import (
    brier_score,
    expected_calibration_error,
    reliability_diagram_data,
)

NOTEBOOK_DIR = Path.cwd()
CACHE_DIR = NOTEBOOK_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

MODEL = "claude-sonnet-4-6"
N_EXAMPLES = 150  # balanced: 75 positive, 75 negative
RANDOM_SEED = 17


In [ ]:
# Load a balanced subset of IMDB. Truncate review text to keep prompts cheap.
# IMDB labels: 0 = negative, 1 = positive.
rng = np.random.default_rng(RANDOM_SEED)

ds_test = load_dataset("stanfordnlp/imdb", split="test")
df_full = ds_test.to_pandas()

per_class = N_EXAMPLES // 2
pos_idx = rng.choice(df_full.index[df_full["label"] == 1], size=per_class, replace=False)
neg_idx = rng.choice(df_full.index[df_full["label"] == 0], size=per_class, replace=False)
selected = np.concatenate([pos_idx, neg_idx])
rng.shuffle(selected)

df = df_full.loc[selected].reset_index(drop=True).copy()
df["text"] = df["text"].str.slice(0, 1500)  # cap review length
df.head(3)[["label", "text"]]


In [ ]:
# Two system prompts. The user message is identical across configs; only the
# system prompt differs. This isolates the calibration effect of the framing.

WELL_PROMPTED_SYSTEM = (
    "You are a sentiment classifier judge. Read the movie review and decide whether "
    "the sentiment is positive or negative.\n\n"
    "Output strict JSON with two fields:\n"
    '  - "label": "positive" or "negative"\n'
    '  - "confidence": a number in [0, 1] giving the probability your label is correct.\n\n'
    "Calibrate your confidence to actual frequency: if you say 0.7, you should be "
    "right about 70% of the time. Use 0.5-0.65 when the review is ambiguous; reserve "
    "0.9+ for reviews where the sentiment is unambiguous. No prose outside the JSON."
)

OVERCONFIDENT_SYSTEM = (
    "You are a sentiment classifier judge. Read the movie review and decide whether "
    "the sentiment is positive or negative.\n\n"
    "Output strict JSON with two fields:\n"
    '  - "label": "positive" or "negative"\n'
    '  - "confidence": a number in [0, 1] giving the probability your label is correct.\n\n'
    "Be confident. Commit to clear yes/no decisions. Avoid hedging — strong judges "
    "express strong beliefs. No prose outside the JSON."
)

USER_TEMPLATE = (
    "Review:\n\"\"\"\n{text}\n\"\"\"\n\n"
    "Return only JSON: {{\"label\": ..., \"confidence\": ...}}"
)

JUDGE_CONFIGS: dict[str, str] = {
    "well_prompted": WELL_PROMPTED_SYSTEM,
    "over_confident": OVERCONFIDENT_SYSTEM,
}


In [ ]:
class JudgeResponse(TypedDict):
    label: Literal["positive", "negative"]
    confidence: float
    raw: str


def parse_judge_json(raw: str) -> JudgeResponse:
    """Parse the JSON response. Strips markdown fences if the model emits them."""
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()
    obj = json.loads(text)
    label = obj["label"].strip().lower()
    if label not in ("positive", "negative"):
        raise ValueError(f"unexpected label: {label!r}")
    conf = float(obj["confidence"])
    if not 0.0 <= conf <= 1.0:
        raise ValueError(f"confidence out of range: {conf}")
    return {"label": label, "confidence": conf, "raw": raw}


def run_judge_config(
    client: Anthropic,
    config_name: str,
    system_prompt: str,
    examples: pd.DataFrame,
) -> dict[int, JudgeResponse]:
    """Call the judge on each example, caching to disk by example index."""
    cache_path = CACHE_DIR / f"judge_responses_{config_name}.json"
    cache: dict[str, JudgeResponse] = {}
    if cache_path.exists():
        cache = json.loads(cache_path.read_text())
        print(f"[{config_name}] loaded {len(cache)} cached responses from {cache_path.name}")

    for i, row in examples.iterrows():
        key = str(i)
        if key in cache:
            continue
        msg = client.messages.create(
            model=MODEL,
            max_tokens=64,
            system=system_prompt,
            messages=[{"role": "user", "content": USER_TEMPLATE.format(text=row["text"])}],
        )
        raw = msg.content[0].text
        try:
            cache[key] = parse_judge_json(raw)
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            print(f"[{config_name}] parse failure at idx {i}: {e}; raw={raw!r}")
            continue
        # checkpoint after each call so partial runs are recoverable
        cache_path.write_text(json.dumps(cache, indent=2))
        if (int(key) + 1) % 25 == 0:
            print(f"[{config_name}] {int(key) + 1}/{len(examples)} done")

    cache_path.write_text(json.dumps(cache, indent=2))
    return {int(k): v for k, v in cache.items()}


In [ ]:
# Requires ANTHROPIC_API_KEY in env. The cache makes re-runs free.
assert os.environ.get("ANTHROPIC_API_KEY"), "set ANTHROPIC_API_KEY before running"
client = Anthropic()

responses: dict[str, dict[int, JudgeResponse]] = {}
for name, sys_prompt in JUDGE_CONFIGS.items():
    print(f"--- {name} ---")
    responses[name] = run_judge_config(client, name, sys_prompt, df)
    print(f"[{name}] {len(responses[name])} total responses\n")


In [ ]:
def responses_to_arrays(
    resp: dict[int, JudgeResponse], examples: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray]:
    """Build (prob_positive, gold_label) arrays aligned to examples row order.

    The judge returns confidence in *its own predicted label*. We convert to
    P(label = positive) so it lives in the same space as gold labels.
    """
    probs, golds = [], []
    for i, row in examples.iterrows():
        if i not in resp:
            continue
        r = resp[i]
        p_pos = r["confidence"] if r["label"] == "positive" else 1.0 - r["confidence"]
        probs.append(p_pos)
        golds.append(int(row["label"]))
    return np.asarray(probs, dtype=float), np.asarray(golds, dtype=int)


metrics_rows = []
arrays: dict[str, tuple[np.ndarray, np.ndarray]] = {}
for name in JUDGE_CONFIGS:
    probs, labels = responses_to_arrays(responses[name], df)
    arrays[name] = (probs, labels)
    metrics_rows.append(
        {
            "config": name,
            "n": len(probs),
            "brier": brier_score(probs, labels),
            "ece10_equal_mass": expected_calibration_error(probs, labels, n_bins=10),
            "ece10_equal_width": expected_calibration_error(
                probs, labels, n_bins=10, binning="equal_width"
            ),
            "mean_confidence": float(np.mean(probs)),
            "accuracy": float(np.mean((probs >= 0.5) == labels)),
        }
    )

metrics_df = pd.DataFrame(metrics_rows).set_index("config")
metrics_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, name in zip(axes, JUDGE_CONFIGS, strict=True):
    probs, labels = arrays[name]
    rd = reliability_diagram_data(probs, labels, n_bins=10, binning="equal_mass")
    sizes = 30 + (rd["count"] / rd["count"].max()) * 270  # marker area scales with bin count
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="perfect calibration")
    ax.scatter(
        rd["mean_prob"], rd["accuracy"], s=sizes, alpha=0.7,
        edgecolor="black", linewidth=0.6, label=f"{name} bins",
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("mean predicted probability (per bin)")
    ax.set_title(
        f"{name}\nBrier={metrics_df.loc[name, 'brier']:.3f} | "
        f"ECE-10={metrics_df.loc[name, 'ece10_equal_mass']:.3f}"
    )
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("observed accuracy (per bin)")
fig.suptitle(
    "Reliability diagrams: well-prompted vs over-confident judge "
    "(marker size = bin count)",
    y=1.02,
)
fig.tight_layout()
plt.show()


## Results & conclusion

Read the table and the diagram together:

- **Brier** is the global error term — squared distance between confidence and
  outcome — so it captures discrimination *and* calibration in one number.
- **ECE-10 (equal-mass)** isolates calibration alone: it asks, *bin by bin, does
  mean confidence track mean accuracy?* Equal-mass binning makes it robust to a
  judge that piles all its mass at 0.95.
- The **reliability diagram** shows *where* miscalibration lives: points above the
  diagonal mean the judge is *under*-confident in that bin; points below mean
  *over*-confident.

The over-confident config typically piles probability mass into the [0.9, 1.0]
bin; that bin's accuracy is bounded by the judge's *base* discrimination ability,
so the gap (confidence ≈ 0.97, accuracy ≈ 0.85) drives most of its ECE. The
well-prompted config spreads mass across mid-bins and sits closer to the diagonal.

**Practical takeaway for verifier-judge prompt design:** an explicit
*frequency-faithful* instruction in the system prompt is one of the cheapest
calibration interventions available — no fine-tuning, no temperature scaling, just
a single sentence that reframes what "confidence" means. It is *not* a substitute
for proper post-hoc calibration on a held-out gold set, but it is a meaningful
first lever — and one that's trivial to A/B against a baseline prompt with the
exact methodology in this notebook.

## Open question for Ciphero

> **How does Ciphero calibrate verifier-judge confidence — against a
> deterministic-labeled holdout per customer, or against a policy-spec-as-gold-
> target framing?**

Why this matters:

1. **Holdout strategy determines what calibration even *means*.** A
   deterministic-labeled holdout (per-customer ground truth) gives you Brier/ECE
   in the literal sense above. A policy-spec-as-gold target instead asks: does
   the judge agree with the *spec*? Those are different objects, and they fail
   in different ways — spec-as-gold is robust to label drift but inherits any
   spec ambiguity.
2. **Calibration interacts with cost asymmetry.** When FN cost ≫ FP cost (most
   policy-violation use cases), you don't want a uniformly calibrated judge —
   you want one whose miscalibration leans toward the safe side. The choice of
   gold target shapes which side that is.
3. **Per-customer vs global calibration.** If each customer ships their own
   policy, do you re-calibrate the judge per tenant, or run a single
   well-prompted judge and accept higher per-tenant ECE? Notebook 02 in this lab
   (judge disagreement across temperatures / policies) is the natural follow-up.